In [1]:
import xmlschema
import xml.etree.ElementTree as ET
from typing import Optional, List, Dict, Any
from dataclasses import dataclass
from pathlib import Path
import logging


In [ ]:
@dataclass
class XMLConfig:
    """Configuration for XML processing.
    
    Args:
        schema_url: URL to XSD schema
        namespace: XML namespace
    """
    schema_url: str = "https://insight.cmdgroup.com/schemas/schDataLinkConfV1.4.xsd"
    namespace: str = "http://insight.cmdgroup.com/schemas/schDataLinkConfV1.4.xsd"

class XMLValidator:
    """Handles XML schema validation."""
    
    def __init__(self, config: XMLConfig):
        self.config = config
        self.schema = self._load_schema()
        
    def _load_schema(self) -> Optional[xmlschema.XMLSchema]:
        """Loads XML schema from URL."""
        try:
            return xmlschema.XMLSchema(self.config.schema_url)
        except Exception as e:
            logging.error(f"Schema loading failed: {e}")
            return None
            
    def validate(self, xml_string: str) -> bool:
        """Validates XML against schema."""
        return bool(self.schema and self.schema.is_valid(xml_string))

class XMLProcessor:
    """Handles XML parsing and cleaning."""
    
    def __init__(self, config: XMLConfig):
        self.config = config
        
    def clean_xml(self, root: ET.Element) -> ET.Element:
        """Removes invalid attributes and normalizes XML."""
        for company in root.findall(f'.//{{{self.config.namespace}}}Company'):
            if 'ContactID' in company.attrib:
                del company.attrib['ContactID']
        return root
        
    def parse_xml(self, file_path: Path) -> Optional[ET.Element]:
        """Parses and cleans XML file."""
        try:
            tree = ET.parse(file_path)
            root = tree.getroot()
            ET.register_namespace("", self.config.namespace)
            return self.clean_xml(root)
        except Exception as e:
            logging.error(f"XML parsing failed: {e}")
            return None

class ProjectExtractor:
    """Extracts project data from XML."""
    
    def __init__(self, validator: XMLValidator, processor: XMLProcessor):
        self.validator = validator
        self.processor = processor
        
    def extract_projects(self, xml_file_path: str) -> Optional[List[Dict[str, Any]]]:
        """
        Extracts projects from XML file.
        
        Args:
            xml_file_path: Path to XML file
            
        Returns:
            List of project dictionaries or None if processing fails
        """
        try:
            path = Path(xml_file_path)
            root = self.processor.parse_xml(path)
            if not root:
                return None
                
            xml_string = ET.tostring(root, encoding="unicode")
            if not self.validator.validate(xml_string):
                return None
                
            doc = self.validator.schema.to_dict(
                xml_string, 
                dict_type='list',
                preserve_root=False
            )
            
            # Clean namespace prefixes
            doc = {k.replace('ns0:', ''): v for k, v in doc.items()}
            
            # Extract projects
            projects = doc['Project'] if isinstance(doc['Project'], list) else [doc['Project']]
            
            # Add default values
            for project in projects:
                project.setdefault('Connections', [])
                for conn in project.get('Connections', []):
                    conn.setdefault('Connection', [])
                    
            return projects
            
        except Exception as e:
            logging.error(f"Project extraction failed: {e}")
            return None

def xml_to_json_projects(xml_file_path: str) -> Optional[List[Dict[str, Any]]]:
    """Main function to process XML file and extract projects."""
    config = XMLConfig()
    validator = XMLValidator(config)
    processor = XMLProcessor(config)
    extractor = ProjectExtractor(validator, processor)
    
    return extractor.extract_projects(xml_file_path)




/var/folders/3m/sdszmrr56p7dpw2s22xnh_980000gn/T/ipykernel_69430/3536501183.py:75: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if not root:


Extracted 964 projects


In [ ]:
# Example usage
projects = xml_to_json_projects("data/cc_delta/Delta_1.4_DL_FBMSales_XML_20241207.xml")
if projects:
    print(f"Extracted {len(projects)} projects")

In [5]:
path = Path('data/cc_delta')

for file_path in path.glob("*.xml"):
    projects = xml_to_json_projects(file_path)
    if projects:
        print(f"Extracted {len(projects)} projects from {file_path}")
        

/var/folders/3m/sdszmrr56p7dpw2s22xnh_980000gn/T/ipykernel_69430/3536501183.py:75: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if not root:


Extracted 1052 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250123.xml
Extracted 964 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20241207.xml
Extracted 1065 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250118.xml
Extracted 1163 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250130.xml
Extracted 303 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250119.xml
Extracted 359 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250126.xml
Extracted 593 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20241224.xml
Extracted 1061 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250129.xml
Extracted 945 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250110.xml
Extracted 816 projects from data/cc_delta/Delta_1.4_DL_FBMSales_XML_20250104.xml
